In [6]:
from pairs_trading.data.loaders import load_prices
from pairs_trading.stats.cointegration import EGResult, JohansenResult
from pairs_trading.config import SplitConfig

Prepare dictionary of tickers to test for cointegration

In [7]:
ticker_dict = {
    'metals':["GLD", "IAU", "SLV", "GDX"],
    "US":["SPY", "IVV", "VTI"],
    "fixed":["TLT", "IEF", "AGG"],
    "energy":["XLE", "XOP"],
    "finance": ["XLF", "KBE"], 
    "real_estate": ["VNQ", "REM"]
    }

Create the full df with all tickers in a short (6 month) time frame to ensure that structural drifts do not wash out cointegration

In [8]:
all_tickers = [ticker for tickers in ticker_dict.values() for ticker in tickers]
df = load_prices(all_tickers, SplitConfig.coint_start, SplitConfig.coint_end, refresh=True)

[*********************100%***********************]  16 of 16 completed


### Egle-Granger Tests

In [9]:
from itertools import combinations

for key in ticker_dict.keys():
    pairs = list(combinations(ticker_dict[key], 2))
    print(f"Results for {key}:")
    for pair in pairs:
        result = EGResult(df, pair)
        result.print_results()
    print("-"*40)

Results for metals:
Engle-Granger Result: y = GLD, x=IAU (statistic=-4.7503873688255425, p-value=0.0004611126407914315, cointegrated: True)
Engle-Granger Result: y = GLD, x=SLV (statistic=-3.5079288531324386, p-value=0.031698956356687596, cointegrated: True)
Engle-Granger Result: y = GLD, x=GDX (statistic=-2.0495678428008746, p-value=0.5027350214298942, cointegrated: False)
Engle-Granger Result: y = SLV, x=IAU (statistic=-3.5241447677857636, p-value=0.03031419454494167, cointegrated: True)
Engle-Granger Result: y = GDX, x=IAU (statistic=-1.5449020002765197, p-value=0.7435746006503803, cointegrated: False)
Engle-Granger Result: y = SLV, x=GDX (statistic=-1.8122658165117187, p-value=0.6237452323422121, cointegrated: False)
----------------------------------------
Results for US:
Engle-Granger Result: y = IVV, x=SPY (statistic=-8.287318171214784, p-value=5.694908294970087e-12, cointegrated: True)
Engle-Granger Result: y = VTI, x=SPY (statistic=-3.1415055920287425, p-value=0.08026680103935

### Johansen Test

In [10]:
from itertools import combinations

for key in ticker_dict.keys():
    pairs = list(combinations(ticker_dict[key], 2))
    print(f"Results for {key}:")
    for pair in pairs:
        result = JohansenResult(df, pair)
        print(f"{pair[0]}, {pair[1]}:")
        result.print_results()
    print("-"*40)

Results for metals:
GLD, IAU:
Johansen Result: (statistic: 35.22878778038795, critical values:15.4943, cointegrated: True)
GLD, SLV:
Johansen Result: (statistic: 27.293221979004322, critical values:15.4943, cointegrated: True)
GLD, GDX:
Johansen Result: (statistic: 20.795792737621735, critical values:15.4943, cointegrated: True)
IAU, SLV:
Johansen Result: (statistic: 27.267627807694726, critical values:15.4943, cointegrated: True)
IAU, GDX:
Johansen Result: (statistic: 20.856429423891523, critical values:15.4943, cointegrated: True)
SLV, GDX:
Johansen Result: (statistic: 22.459290786852833, critical values:15.4943, cointegrated: True)
----------------------------------------
Results for US:
SPY, IVV:
Johansen Result: (statistic: 29.58039875218146, critical values:15.4943, cointegrated: True)
SPY, VTI:
Johansen Result: (statistic: 7.311101550716691, critical values:15.4943, cointegrated: False)
IVV, VTI:
Johansen Result: (statistic: 8.213162756304152, critical values:15.4943, cointegrat

Based on these results I will select pairs that pass one of the tests and are not redundant or trivial (as the many metal combinations). Hence, I pick:

- IAU/GDX
- GLD/SLV
- XLF/KBE
- SPY/IVV (follows the same index so profit should be arbitraged away but can be used as a test)
